#### **Fake News Detector**

In [1]:
import requests
import os
from dotenv import load_dotenv
import pandas as pd
import time

status = load_dotenv("../../secrets.env")
if status:
    print("Success: secrets.env loaded!")
else:
    print("Fail: Could not find or read secrets.env.")

Success: secrets.env loaded!


In [2]:
token = os.getenv("HUGGING_FACE_token")
headers = {"Authorization": f"Bearer {token}"}

In [3]:
#url = "https://api-inference.huggingface.co/models/your-username/fake-news-bert"
#url = "https://router.huggingface.co/hf-inference/models/your-username/fake-news-bert"
url = "https://router.huggingface.co/hf-inference/models/hamzab/roberta-fake-news-classification"
#url = "https://router.huggingface.co/hf-inference/models/jy46604790/Fake-News-Bert-Detect"
#url = "https://router.huggingface.co/hf-inference/models/mrm8488/bert-tiny-finetuned-fake-news-detection"

payload = {"inputs": " Top Trump Surrogate BRUTALLY Stabs Him In The Back: â€˜Heâ€™s Patheticâ€™ (VIDEO) It s looking as though Republican presidential candidate Donald Trump is losing support even from within his own ranks. You know things are getting bad when even your top surrogates start turning against you, which is exactly what just happened on Fox News when Newt Gingrich called Trump  pathetic. Gingrich knows that Trump needs to keep his focus on Hillary Clinton if he even remotely wants to have a chance at defeating her. However, Trump has hurt feelings because many Republicans don t support his sexual assault against women have turned against him, including House Speaker Paul Ryan (R-WI). So, that has made Trump lash out as his own party.Gingrich said on Fox News: Look, first of all, let me just say about Trump, who I admire and I ve tried to help as much as I can. There s a big Trump and a little Trump. The little Trump is frankly pathetic. I mean, he s mad over not getting a phone call? Trump s referring to the fact that Paul Ryan didn t call to congratulate him after the debate. Probably because he didn t win despite what Trump s ego tells him.Gingrich also added: Donald Trump has one opponent. Her name is Hillary Clinton. Her name is not Paul Ryan. It s not anybody else. Trump doesn t seem to realize that the person he should be mad at is himself because he truly is his own worst enemy. This will ultimately lead to his defeat and he will have no one to blame but himself.Watch here via Politico:Featured Photo by Joe Raedle/Getty Images"}

response = requests.post(url, headers=headers, json=payload)

# Check if the request was successful
if response.status_code == 200:
    prediction = response.json()
    print(f"Prediction: {prediction}")
else:
    print(f"Error {response.status_code}: {response.text}")


Prediction: [[{'label': 'FAKE', 'score': 0.9999797344207764}, {'label': 'TRUE', 'score': 2.029385905188974e-05}]]


In [4]:
#df = pd.read_csv("./data2/cleaned_file_totally.csv")
df = pd.read_csv("./data4/WELFake_clean.csv")

In [5]:
df

,Text,label
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,Fake
1,Did they post their votes for Hillary already?,Fake
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,Fake
3,"Bobby Jindal, raised Hindu, uses story of Chri...",Real
4,SATAN 2: Russia unvelis an image of its terrif...,Fake
...,...,...
72129,Russians steal research on Trump in hack of U....,Real
72130,WATCH: Giuliani Demands That Democrats Apologi...,Fake
72131,Migrants Refuse To Leave Train At Refugee Camp...,Real
72132,Trump tussle gives unpopular Mexican leader mu...,Real


In [6]:
df['length'] = df.iloc[:, 0].astype(str).str.len()

In [7]:
df['truncated_text'] = df['Text'].str[:2200]
df.head(1)

,Text,label,length,truncated_text
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,Fake,5180,LAW ENFORCEMENT ON HIGH ALERT Following Threat...


In [8]:
# result: [[{'label': 'FAKE', 'score': 0.9999797344207764}, {'label': 'TRUE', 'score': 2.029385905188974e-05}]]
def classify(text, retries=15, wait=20):
    payload = {"inputs": str(text)}
    for attempt in range(retries):
        try:
            response = requests.post(url, headers=headers, json=payload, timeout=30)
            if response.status_code == 200:
                results = response.json()[0]
                best   = max(results, key=lambda x: x["score"])
                label  = best["label"].upper()
                pred_label = "REAL" if label == "TRUE" else label
                pred_prob = best["score"]
                return pred_label, pred_prob
            elif response.status_code == 503:
                print(f"  Model loading, waiting {wait}s... (attempt {attempt+1})")
                time.sleep(wait)
            elif response.status_code == 400:
                # Bad request — no point retrying with same text, skip immediately
                print(f"  [Attempt {attempt+1}/{retries}] HTTP 400 Bad Request: {response.text[:120]}")
                print(f"  Truncating further and retrying...")
                text = text[:2200-(250*(attempt+1))]  # shrink text each retry
                payload = {"inputs": text}
                time.sleep(2)
            else:
                print(f"  HTTP {response.status_code}: {response.text[:120]}")
                return "ERROR", 0.0
        except Exception as e:
            print(response.status_code)
            print(f"  [Attempt {attempt+1}/{retries}] Exception: {e} — retrying in {wait}s...")
            time.sleep(wait)
    return "ERROR"

In [9]:
predictions = []
probs = []

for i, row in df.iterrows():
    text  = row["truncated_text"]
    label_p, prob_p = classify(text)
    predictions.append(label_p)
    probs.append(prob_p)
    print(f"[{i+1}/{len(df)}] → {row['label']} > {label_p} {label_p} {prob_p * 100:.3f}%| {str(text)[:80]}")
    time.sleep(0.5)  # polite API usage

df["Prediction"] = predictions
df.head(10)

  [Attempt 1/15] HTTP 400 Bad Request: {"error":"The expanded size of the tensor (558) must match the existing size (514) at non-singleton dimension 1.  Target
  Truncating further and retrying...
[1/72134] → Fake > FAKE FAKE 99.998%| LAW ENFORCEMENT ON HIGH ALERT Following Threats Against Cops And Whites On 9-11B
[2/72134] → Fake > FAKE FAKE 99.948%| Did they post their votes for Hillary already?
[3/72134] → Fake > FAKE FAKE 99.998%| UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MOST CHARLOTTE RIOTERS WERE “PEACEFU
[4/72134] → Real > REAL REAL 99.996%| Bobby Jindal, raised Hindu, uses story of Christian conversion to woo evangelica
[5/72134] → Fake > REAL REAL 99.985%| SATAN 2: Russia unvelis an image of its terrifying new ‘SUPERNUKE’ – Western wor
[6/72134] → Fake > FAKE FAKE 99.994%| About Time! Christian Group Sues Amazon and SPLC for Designation as Hate Group A
[7/72134] → Fake > FAKE FAKE 99.998%| DR BEN CARSON TARGETED BY THE IRS: “I never had an audit until I spoke at the Na
[8/7

UnboundLocalError: local variable 'response' referenced before assignment

In [ ]:
df